In [2]:
import numpy as np
import pandas as pd
import re
import statsmodels.api as sm
import pandas as pd

dt = 1.0  # day

# ----------------------------
# Load spot data
# ----------------------------
# ----------------------------
# Read txt file
# ----------------------------
with open("../data/THE_gas.txt", "r", encoding="utf-8") as f:
    txt = " ".join(f.readlines())

# ----------------------------
# Extract dates
# ----------------------------
dates_str = re.search(r"labels:\s*\[(.*?)\]", txt).group(1)
dates = [d.replace("'", "").strip() for d in dates_str.split(",")]

# ----------------------------
# Extract prices
# ----------------------------
prices_str = re.search(r"data:\s*\[(.*?)\]", txt).group(1)
prices = [p.strip() for p in prices_str.split(",")]

# Convert empty strings to NaN
prices = [np.nan if p == "" else float(p) for p in prices]

# ----------------------------
# Construct dataframe
# ----------------------------
data = pd.DataFrame({
    "date": pd.to_datetime(dates, format="%d.%m.%Y"),
    "spot_price": prices
})

# ----------------------------
# Filter data
# ----------------------------
data = (
    data
    .dropna(subset=["spot_price"])
    .loc[(data["date"] >= "2024-01-02") & (data["date"] <= "2025-12-31")]
    .sort_values("date")
    .reset_index(drop=True)
)

# ----------------------------
# Joint estimation of seasonality + AR(1)
# ----------------------------
data["t"] = (data["date"] - data["date"].iloc[0]).dt.days.astype(float)
data["X"] = np.log(data["spot_price"].astype(float))
data["X_lag"] = data["X"].shift(1)

reg_df = data.dropna(subset=["X", "X_lag"]).copy()

t = reg_df["t"].to_numpy()
X_lag = reg_df["X_lag"].to_numpy()
y = reg_df["X"].to_numpy()

# Seasonal basis functions for \tilde{\mu}(t)
sin1 = np.sin(2.0 * np.pi * t / 365.0)
cos1 = np.cos(2.0 * np.pi * t / 365.0)

# Regression:
# X_{t+1} = beta * X_t + c0 + c1*sin(...) + c2*cos(...) + eps_t
# where c_j = (1 - beta) * a_j
Xreg = np.column_stack([X_lag, sin1, cos1])
Xreg = sm.add_constant(Xreg)

fit_joint = sm.OLS(y, Xreg).fit()

const_hat = float(fit_joint.params[0])
b_hat = float(fit_joint.params[1])
c1_hat = float(fit_joint.params[2])
c2_hat = float(fit_joint.params[3])

sigma_eta_hat = float(np.std(fit_joint.resid, ddof=1))

# Recover continuous-time parameters
kappa_hat = -np.log(b_hat) / dt

sigma_hat = np.sqrt(
    (sigma_eta_hat**2) * 2.0 * kappa_hat /
    (1.0 - np.exp(-2.0 * kappa_hat * dt))
)

# Recover seasonal mean coefficients a_j from c_j = (1 - beta) * a_j
denom = 1.0 - b_hat
a0_hat = const_hat / denom
a1_hat = c1_hat / denom
a2_hat = c2_hat / denom

params = pd.DataFrame({
    "a0_hat": [a0_hat],
    "a1_hat": [a1_hat],
    "a2_hat": [a2_hat]
})

params.to_csv("../data/seasonal_params.csv", index=False)
